In [1]:
import glob
import pandas as pd

csv_files = glob.glob('*.csv')

for filename in csv_files:
    print(f"📄 文件名: {filename}")
    try:
        # 只读取前 2 行 (nrows=1 读取数据行，header默认是第一行)
        # 注意：pd.read_csv 默认把第一行当做表头，nrows=1 会读入表头 + 1行数据
        df = pd.read_csv(filename, nrows=1)
        
        # 打印表头（列名）
        print(f"   第一行 (列名): {list(df.columns)}")
        
        # 打印第一行数据 (如果文件不为空)
        if not df.empty:
            print(f"   第二行 (数值): {df.iloc[0].tolist()}")
            
    except Exception as e:
        print(f"   读取出错: {e}")
    print("-" * 40)

📄 文件名: AD.csv
   第一行 (列名): ['path', 'filename', 'age', 'gender', 'education', 'hispanic', 'race', 'apoe', 'NC', 'MCI', 'DE', 'COG', 'AD', 'PD', 'FTD', 'VD', 'DLB', 'PDD', 'ADD', 'OTHER', 'mmse', 'cdr', 'cdrSum', 'Tesla', 'trailA', 'trailB', 'lm_imm', 'lm_del', 'boston', 'animal', 'vege', 'digitB', 'digitBL', 'digitF', 'digitFL', 'npiq_DEL', 'npiq_HALL', 'npiq_AGIT', 'npiq_DEPD', 'npiq_ANX', 'npiq_ELAT', 'npiq_APA', 'npiq_DISN', 'npiq_IRR', 'npiq_MOT', 'npiq_NITE', 'npiq_APP', 'faq_BILLS', 'faq_TAXES', 'faq_SHOPPING', 'faq_GAMES', 'faq_STOVE', 'faq_MEALPREP', 'faq_EVENTS', 'faq_PAYATTN', 'faq_REMDATES', 'faq_TRAVEL', 'his_CVHATT', 'his_PSYCDIS', 'his_Alcohol', 'his_SMOKYRS', 'his_PACKSPER', 'his_NACCFAM', 'his_CBSTROKE', 'his_HYPERTEN', 'his_DEPOTHR', 'gds', 'moca']
   第二行 (数值): ['C:\\Users\\86137\\ClinicRoom\\ADNI\\ad_nii_KG/', 'ADNI_109_S_1157_MR_MPR__GradWarp__B1_Correction__N3__Scaled_Br_20070808201650854_S24711_I66158.nii', nan, 'female', -4, -4, nan, 1.0, 0, 0, 1, 2, 1, 0.0, nan, 

In [3]:
import pandas as pd

# 1. 读取两个三元组文件
df_adni = pd.read_csv("adni_knowledge_triplets.csv")
df_prime = pd.read_csv("primekg_ad_only.csv")

# 2. 提取 ADNI 侧试图连接的 "PrimeKG" 开头的尾实体
# (代码中病人连接到 -> PrimeKG:xxxx)
adni_targets = set(df_adni[df_adni['tail'].str.startswith('PrimeKG:')]['tail'].unique())

# 3. 提取 PrimeKG 侧实际存在的实体 (加上前缀以匹配你的格式)
# 注意：你的 primekg_ad_only.csv 里的 x_name 是原始名字 (如 "DHX9")
# 而你的代码里给 ADNI 加上了 "PrimeKG:" 前缀
# 所以我们要看 PrimeKG 文件里的名字是否包含 ADNI 想连的那些名字
prime_nodes = set("PrimeKG:" + df_prime['x_name'].astype(str)) | \
              set("PrimeKG:" + df_prime['y_name'].astype(str))

# 4. 求交集：看有多少能连上
overlap = adni_targets.intersection(prime_nodes)
missing = adni_targets - prime_nodes

print(f"📊 连通性分析:")
print(f"   - ADNI 试图连接的 PrimeKG 节点数: {len(adni_targets)}")
print(f"   - 实际在 PrimeKG 子图中存在的节点数: {len(overlap)}")
print(f"   - ❌ 断链节点 (ADNI指涉了但PrimeKG里没有): {len(missing)}")

if len(missing) > 0:
    print(f"\n⚠️ 警告：以下关键节点可能名字不匹配，导致图谱断裂！")
    print(list(missing)[:10]) # 打印前10个看看
else:
    print("\n✅ 完美！所有桥接节点都已对齐。")

📊 连通性分析:
   - ADNI 试图连接的 PrimeKG 节点数: 2460
   - 实际在 PrimeKG 子图中存在的节点数: 2459
   - ❌ 断链节点 (ADNI指涉了但PrimeKG里没有): 1

⚠️ 警告：以下关键节点可能名字不匹配，导致图谱断裂！
['PrimeKG:Healthy']
